In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/

In [2]:
IMG_HEIGHT = 100
IMG_WIDTH = 100
BATCH_SIZE = 16
NUM_CLASSES = 4

In [6]:
DATASET_DIR = r"D:\AQI_ESTIMATION\model\oversampled_dataset"

In [7]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

In [8]:
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 1720 images belonging to 4 classes.


In [9]:
val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 429 images belonging to 4 classes.


In [10]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

In [11]:
for layer in base_model.layers:
    layer.trainable = False

In [12]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

In [13]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,782,660 (105.98 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [15]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

Epoch 1/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.4622 - loss: 1.7654 - val_accuracy: 0.4918 - val_loss: 1.3579
Epoch 2/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.4930 - loss: 1.3399 - val_accuracy: 0.4918 - val_loss: 1.3265
Epoch 3/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 93s 856ms/step - accuracy: 0.4930 - loss: 1.3128 - val_accuracy: 0.4918 - val_loss: 1.3014
Epoch 4/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 79s 729ms/step - accuracy: 0.4930 - loss: 1.2908 - val_accuracy: 0.4918 - val_loss: 1.2823
Epoch 5/5
108/108 ━━━━━━━━━━━━━━━━━━━━ 47s 439ms/step - accuracy: 0.4930 - loss: 1.2744 - val_accuracy: 0.4918 - val_loss: 1.2679


In [16]:
model.save("aqi_model_oversampled.h5")
print("Oversampling model saved successfully")

Oversampling model saved successfully
